# Kolmogorov flow: 2–6× downsampling

Five seeds [2, 42, 123, 468, 500] select time windows and sampling noise. All diffusion methods use 100 reverse steps. Baseline checkpoints match each downsampling factor.

Run cells in order. Sampling counts are reverse-diffusion steps; checkpoint numbers identify training state.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'kolmogorov_downsampling'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)
import torch.nn.functional as F


In [ ]:
import os
import sys
import json
import importlib
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import namedtuple
from tqdm import tqdm
from einops import rearrange
from sampler.utils import *


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
from sampler.utils import *
import sampler.utils
from trainer.utils import restore_checkpoint
from models import get_model, get_ema
from sampler import VESDE
from models.fno import FNO2d
from models.lno import LNO2d


In [ ]:
outer_loop = 100
denoising_steps = outer_loop
data_name = 'kol'
version = 'v1'
opt_path = './results/kol_v1_finetune_FPE/config.json'
ckpt_path = './results/kol_v1_finetune_FPE/checkpoint_517.pth'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt_dict = json.load(f)
    opt_dict['num_channels'] = opt_dict['num_components'] + opt_dict['num_conditions']
opt_dict['results_path'] = str(Path(opt_path).parent)
opt_dict['device'] = device
OPT_class = namedtuple('OPT_class', opt_dict.keys())
opt = OPT_class(**opt_dict)
print('>>> Loading SDE Model...')
net = get_model(opt)
net = nn.DataParallel(net, device_ids=[0] if torch.cuda.is_available() else [])
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
net.eval()
print(f"SDE Model loaded from epoch {state['epoch']}")
data_test_path = './data/kolmogorov/kolmogorov_flow_test.npy'
scalar_path = opt.results_path + '/scalar.npy'
(mean, std) = np.load(scalar_path)
scalar = lambda x: (x - mean) / std
scalar_inv = lambda x: x * std + mean
data_raw = np.load(data_test_path)
x_test = data_raw.reshape(5, -1, 400, 64, 64, 2)
sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps)


In [ ]:
data_test_path = './data/kolmogorov/kolmogorov_flow_test.npy'
scalar_path = opt.results_path + '/scalar.npy'
data_raw = np.load(data_test_path)
rey = np.array([50, 125, 575, 1100, 1500])
sigma = np.array([2, 4, 6, 8])
(_, z) = np.meshgrid(np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), indexing='ij')
x_test = data_raw.reshape(5, -1, 400, 64, 64, 2)
(mean, std) = np.load(scalar_path)

def scalar(x):
    return (x - mean) / std

def scalar_inv(x):
    return x * std + mean
print(f'Data loaded: {x_test.shape}')


In [ ]:
def get_grid(shape, device):
    (batchsize, size_x, size_y) = (shape[0], shape[1], shape[2])
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)


In [ ]:
def run_evaluation(scale, current_seed):
    print(f'\n=== Evaluation: Scale={scale}, Seed={current_seed} ===')
    m = 2
    T = opt.num_frames
    T_prime_target = 100
    B = int(np.ceil((T_prime_target - m) / (T - m)))
    T_prime = B * (T - m) + m
    n_samples = 1
    seed_samples = current_seed
    samples = []
    num_re = len(x_test)
    num_k = len(x_test[0])
    (s_re, s_k) = (4, 1)
    for (i_re, tr_k) in enumerate(x_test):
        for (i_k, tr) in enumerate(tr_k):
            rnd = np.random.RandomState(seed_samples + i_re * num_re + i_k)
            i_steps = rnd.choice(len(tr) - T - T_prime - 10, n_samples, replace=False) + 10
            if i_re == s_re and i_k == s_k:
                for i in i_steps:
                    single_sample = x_test[i_re, i_k, i:i + T_prime].transpose(0, 3, 1, 2)
                    f = np.sin(sigma[i_k] * z)[np.newaxis, np.newaxis].repeat(T_prime, axis=0)
                    r = 1.0 / rey[i_re] * 100.0 * np.ones_like(f)
                    samples.append(np.concatenate([single_sample, f, r], axis=1))
    samples = np.stack(samples, axis=0)
    print(f'  -> Sample shape: {samples.shape}, T_prime: {T_prime}')
    samples_scale = samples.copy()
    samples_scale[:, :, :opt.num_components] = scalar(samples_scale[:, :, :opt.num_components])
    y = samples_scale[:, :T_prime, :2, ::scale, ::scale]
    transform_gen = lambda x: x[:, :T_prime, :2, ::scale, ::scale]
    sparsity = 1 - 1 / scale ** 2
    alpha = 0.5 / np.sqrt(1 - sparsity)
    beta = 0.4
    gamma = 0.5
    reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
    latent_mask = torch.ones([opt.num_frames, 1, 1, 1]).float().to(device)
    obs_mask = torch.zeros([opt.num_frames, 1, 1, 1]).float().to(device)
    frame_indices = torch.arange(opt.num_frames).to(device)
    net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
    print('  -> Running SDE...')
    setup_seed(current_seed)
    x_pred = s3gm_sample_2d(opt, net_fn, sde, y, transform_gen, alpha=alpha, beta=beta, gamma=gamma, T_prime_y=T_prime, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False)
    print('  -> Running ISS...')
    setup_seed(current_seed)
    x_pred1 = s3gm_sample_2d_ISS_kol(opt, net_fn, sde1, y, transform_gen, alpha=1 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, reg_coef_pa=20 * T_prime, reg_coef_ar=0, denoising_steps=denoising_steps, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
    print('  -> Running LNO...')
    x_pred2 = np.zeros_like(x_pred)
    try:
        lno_path = f'results_LNO/lno_reconstruction_{scale}x.pth'
        lno_model = LNO2d(in_channels=20, out_channels=20, width=8, modes1=16, modes2=16, tx=torch.linspace(0, 1, 64).unsqueeze(0).to(device), ty=torch.linspace(0, 1, 64).unsqueeze(0).to(device)).to(device)
        if os.path.exists(lno_path):
            lno_model.load_state_dict(torch.load(lno_path, map_location=device))
            lno_model.eval()
            gt_input = torch.tensor(samples_scale[:, :, :2], dtype=torch.float).to(device)
            SEQ_LEN = 10
            INPUT_RES = scale
            pred_chunks = []
            for i in range(int(np.ceil(T_prime / SEQ_LEN))):
                chunk = gt_input[:, i * SEQ_LEN:(i + 1) * SEQ_LEN]
                pad = 0
                if chunk.shape[1] < SEQ_LEN:
                    pad = SEQ_LEN - chunk.shape[1]
                    chunk = torch.cat([chunk, torch.zeros(1, pad, 2, 64, 64).to(device)], dim=1)
                x_low = chunk[:, :, :, ::INPUT_RES, ::INPUT_RES]
                x_in = F.interpolate(x_low.reshape(-1, 2, x_low.shape[3], x_low.shape[4]), size=(64, 64), mode='bilinear', align_corners=True)
                x_in_lno = x_in.reshape(1, SEQ_LEN * 2, 64, 64)
                with torch.no_grad():
                    out = lno_model(x_in_lno).reshape(1, SEQ_LEN, 2, 64, 64)
                if pad > 0:
                    out = out[:, :-pad]
                pred_chunks.append(out)
            x_pred2[:, :, :2] = torch.cat(pred_chunks, dim=1).cpu().numpy()
        else:
            raise FileNotFoundError(lno_path)
    except Exception as e:
        raise
    print('  -> Running FNO...')
    x_pred3 = np.zeros_like(x_pred)
    try:
        fno_path = f'results_FNO/fno_reconstruction_{scale}x.pth'
        fno_model = FNO2d(num_channels=2, out_channels=20, modes1=16, modes2=16, width=64, initial_step=10).to(device)
        if os.path.exists(fno_path):
            fno_model.load_state_dict(torch.load(fno_path, map_location=device))
            fno_model.eval()
            gt_input = torch.tensor(samples_scale[:, :, :2], dtype=torch.float).permute(0, 1, 3, 4, 2).to(device)
            SEQ_LEN = 10
            INPUT_RES = scale
            pred_chunks = []
            for i in range(int(np.ceil(T_prime / SEQ_LEN))):
                chunk = gt_input[:, i * SEQ_LEN:(i + 1) * SEQ_LEN]
                pad = 0
                if chunk.shape[1] < SEQ_LEN:
                    pad = SEQ_LEN - chunk.shape[1]
                    chunk = torch.cat([chunk, torch.zeros(1, pad, 64, 64, 2).to(device)], dim=1)
                x_low = chunk[:, :, ::INPUT_RES, ::INPUT_RES, :]
                x_interp = F.interpolate(x_low.permute(0, 1, 4, 2, 3).reshape(-1, 2, x_low.shape[2], x_low.shape[3]), size=(64, 64), mode='bilinear', align_corners=True)
                x_in = x_interp.reshape(1, SEQ_LEN, 2, 64, 64).permute(0, 3, 4, 1, 2).reshape(1, 64, 64, SEQ_LEN * 2)
                grid = get_grid(x_in.shape, device)
                with torch.no_grad():
                    out = fno_model(x_in, grid).squeeze(-2)
                    out = out.reshape(1, 64, 64, SEQ_LEN, 2).permute(0, 3, 4, 1, 2)
                if pad > 0:
                    out = out[:, :-pad]
                pred_chunks.append(out)
            x_pred3[:, :, :2] = torch.cat(pred_chunks, dim=1).cpu().numpy()
        else:
            raise FileNotFoundError(fno_path)
    except Exception as e:
        raise
    print('Calculating reconstruction errors...')
    x_pred[:, :, :opt.num_components] = scalar_inv(x_pred[:, :, :opt.num_components])
    x_pred1[:, :, :opt.num_components] = scalar_inv(x_pred1[:, :, :opt.num_components])
    x_pred2[:, :, :opt.num_components] = scalar_inv(x_pred2[:, :, :opt.num_components])
    x_pred3[:, :, :opt.num_components] = scalar_inv(x_pred3[:, :, :opt.num_components])
    vor_pred = vor_cal_batch(np.concatenate([x_pred[0, :, 1:2], x_pred[0, :, :1]], axis=1), opt.image_size, [0, 2 * np.pi])
    vor_ref = vor_cal_batch(np.concatenate([samples[0, :, 1:2], samples[0, :, :1]], axis=1), opt.image_size, [0, 2 * np.pi])
    vor_pred1 = vor_cal_batch(np.concatenate([x_pred1[0, :, 1:2], x_pred1[0, :, :1]], axis=1), opt.image_size, [0, 2 * np.pi])
    vor_pred2 = vor_cal_batch(np.concatenate([x_pred2[0, :, 1:2], x_pred2[0, :, :1]], axis=1), opt.image_size, [0, 2 * np.pi])
    vor_pred3 = vor_cal_batch(np.concatenate([x_pred3[0, :, 1:2], x_pred3[0, :, :1]], axis=1), opt.image_size, [0, 2 * np.pi])
    errors = cal_rmse(samples[0, :, 0:2, :], x_pred[0, :, 0:2, :], normalize=True)
    mean_error = np.mean(errors)
    print(f'mean_error (SDE): {mean_error:.4f}')
    errors1 = cal_rmse(samples[0, :, 0:2, :], x_pred1[0, :, 0:2, :], normalize=True)
    mean_error1 = np.mean(errors1)
    print(f'mean_error1 (ISS): {mean_error1:.4f}')
    errors2 = cal_rmse(samples[0, :, 0:2, :], x_pred2[0, :, 0:2, :], normalize=True)
    mean_error2 = np.mean(errors2)
    print(f'mean_error2 (LNO): {mean_error2:.4f}')
    errors3 = cal_rmse(samples[0, :, 0:2, :], x_pred3[0, :, 0:2, :], normalize=True)
    mean_error3 = np.mean(errors3)
    print(f'mean_error3 (FNO): {mean_error3:.4f}')
    num_T = x_pred.shape[1]
    print(f'Number of generated snapshots: {num_T}')
    torch.cuda.empty_cache()
    return {'SDE': mean_error, 'ISS': mean_error1, 'LNO': mean_error2, 'FNO': mean_error3}


In [ ]:
scales = [2, 3, 4, 5, 6]
seeds = [2, 42, 123, 468, 500]
results = {'SDE': {s: [] for s in scales}, 'ISS': {s: [] for s in scales}, 'LNO': {s: [] for s in scales}, 'FNO': {s: [] for s in scales}}
for s in scales:
    for seed in seeds:
        metrics = run_evaluation(scale=s, current_seed=seed)
        for method in results.keys():
            results[method][s].append(metrics[method])


In [ ]:
print('\n' + '=' * 60)
print(f"{'Method':<8} | {'Scale':<6} | {'Mean nRMSE':<12} | {'Std Dev':<12}")
print('-' * 60)
method_print_order = ['LNO', 'FNO', 'ISS', 'SDE']
for method in method_print_order:
    if method not in results:
        continue
    for s in scales:
        data = np.array(results[method][s])
        print(f'{method:<8} | {s:<6} | {np.mean(data):.6f}     | {np.std(data):.6f}')
    print('-' * 60)


## Save trial results


In [ ]:
rows = [{'method': method, 'scale': scale, 'seed': seed, 'nrmse': float(value)} for (method, by_scale) in results.items() for (scale, values) in by_scale.items() for (seed, value) in zip(seeds, values)]
pd.DataFrame(rows).to_csv(OUTPUT / 'downsampling_trials.csv', index=False)
(fig, ax) = plt.subplots(figsize=(7, 4))
for method in results:
    ax.errorbar(scales, [np.mean(results[method][s]) for s in scales], yerr=[np.std(results[method][s]) for s in scales], marker='o', capsize=3, label=method)
ax.set(xlabel='Spatial downsampling factor', ylabel='Velocity nRMSE')
ax.legend()
format_current_figure()
fig.savefig(OUTPUT / 'downsampling_nrmse.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
